# F1 Data Exploration
Row counts, sample queries, referential integrity checks.

In [ ]:
from sqlalchemy import text
from backend.database import engine

tables = [
    'seasons', 'circuits', 'constructors', 'drivers', 'races', 'sessions',
    'driver_race_entries', 'race_results', 'qualifying_results', 'laps',
    'pit_stops', 'tyre_stints', 'weather', 'telemetry_samples',
    'race_control_messages', 'team_radio'
]

with engine.connect() as conn:
    for t in tables:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        print(f'{t:30s} {count:>10,}')

In [ ]:
import pandas as pd

# Fastest laps for a session
query = """
SELECT d.code, d.last_name, l.lap_number, l.lap_time_ms,
       l.sector1_ms, l.sector2_ms, l.sector3_ms, l.compound
FROM laps l
JOIN drivers d ON d.id = l.driver_id
JOIN sessions s ON s.id = l.session_id
JOIN races r ON r.id = s.race_id
WHERE r.name LIKE '%Bahrain%' AND s.session_type = 'R'
ORDER BY l.lap_time_ms
LIMIT 20
"""
pd.read_sql(query, engine)

In [ ]:
# Tyre compound distribution
query = "SELECT DISTINCT compound FROM tyre_stints ORDER BY compound"
pd.read_sql(query, engine)

In [ ]:
# Referential integrity check - orphaned foreign keys
integrity_checks = [
    "SELECT COUNT(*) FROM laps l LEFT JOIN sessions s ON s.id = l.session_id WHERE s.id IS NULL",
    "SELECT COUNT(*) FROM race_results rr LEFT JOIN races r ON r.id = rr.race_id WHERE r.id IS NULL",
    "SELECT COUNT(*) FROM telemetry_samples t LEFT JOIN sessions s ON s.id = t.session_id WHERE s.id IS NULL",
]
with engine.connect() as conn:
    for q in integrity_checks:
        orphans = conn.execute(text(q)).scalar()
        print(f'Orphans: {orphans} — {q[:80]}')